<a href="https://colab.research.google.com/github/UntalHugo/universidad-ia/blob/main/clase-01-rag-ventas/02_agente_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Agente RAG con Mistral + LangChain
Objetivo: crear un agente que consulte los datos de ventas y responda preguntas.

Celda 1 — Instalaciones


In [ ]:
!pip install pandas langchain langchain-experimental tabulate langchain-ollama langchain-mistralai

Celda 2 — Configurar el Secret en Colab


Antes de escribir código, hay que cargar la API key. Seguí estos pasos:
1. En el panel izquierdo buscá el ícono de 🔑 llave
2. Clic en "Add new secret"
3. Completá así:
Name  → MISTRAL_API_KEY
Value → (pegás tu API key de Mistral acá)
4. Activá el toggle "Notebook access"

Celda 3 — Cargar la API key

In [ ]:
from google.colab import userdata
import os

os.environ["MISTRAL_API_KEY"] = userdata.get("MISTRAL_API_KEY")
print("✅ API Key cargada correctamente")

✅ API Key cargada correctamente


Celda 4 — Testing de conexión con Mistral

In [ ]:
from langchain_mistralai import ChatMistralAI

# Inicializamos el modelo
# temperature=0 → respuestas deterministas, sin creatividad
# ideal para consultas sobre datos donde queremos precisión
llm = ChatMistralAI(
    model="mistral-small-latest",
    temperature=0
)

# Test simple para verificar que la conexión funciona
respuesta = llm.invoke("Respondé solo esto: ¿estás funcionando?")
print(respuesta.content)

Sí, estoy funcionando correctamente. ¿En qué puedo ayudarte?


Celda 5 — Cargar los datos normalizados


In [ ]:
import pandas as pd

# Creamos datos de ejemplo para poder ejecutar el agente
data_clientes = {'id_cliente': [1, 2], 'nombre': ['Juan Perez', 'Maria Lopez']}
data_productos = {'id_producto': [101, 102], 'producto': ['Laptop', 'Mouse']}
data_ordenes = {'id_orden': [501, 502], 'id_cliente': [1, 2], 'fecha': ['2023-01-01', '2023-01-02']}
data_items = {'id_orden': [501, 502], 'id_producto': [101, 102], 'cantidad': [1, 5], 'precio': [1000, 20]}

# Guardamos los CSVs que el código espera encontrar
pd.DataFrame(data_clientes).to_csv('dim_clientes.csv', index=False)
pd.DataFrame(data_productos).to_csv('dim_productos.csv', index=False)
pd.DataFrame(data_ordenes).to_csv('fact_ordenes.csv', index=False)
pd.DataFrame(data_items).to_csv('fact_items.csv', index=False)

print("✅ Archivos CSV creados exitosamente para la demostración.")

✅ Archivos CSV creados exitosamente para la demostración.


In [ ]:
import pandas as pd
import os

# Verificamos que los archivos existan antes de cargar
files = ["dim_clientes.csv", "dim_productos.csv", "fact_ordenes.csv", "fact_items.csv"]
if all(os.path.exists(f) for f in files):
    dim_clientes  = pd.read_csv("dim_clientes.csv")
    dim_productos = pd.read_csv("dim_productos.csv")
    fact_ordenes  = pd.read_csv("fact_ordenes.csv")
    fact_items    = pd.read_csv("fact_items.csv")

    print("✅ Tablas cargadas exitosamente:")
    print(f"   dim_clientes  → {dim_clientes.shape[0]} filas")
    print(f"   dim_productos → {dim_productos.shape[0]} filas")
    print(f"   fact_ordenes  → {fact_ordenes.shape[0]} filas")
    print(f"   fact_items    → {fact_items.shape[0]} filas")
else:
    print("❌ Error: Algunos archivos CSV aún no han sido creados. Ejecuta la celda anterior.")

✅ Tablas cargadas exitosamente:
   dim_clientes  → 2 filas
   dim_productos → 2 filas
   fact_ordenes  → 2 filas
   fact_items    → 2 filas


Celda 6 — Importaciones


In [ ]:
from langchain_experimental.agents import create_pandas_dataframe_agent
from langchain_mistralai import ChatMistralAI

# Ya no importamos AgentType, pasamos el string directamente
print("✅ Importaciones listas")

✅ Importaciones listas


Celda 7 — Crear el agente


In [ ]:
agente = create_pandas_dataframe_agent(
    llm=llm,
    df=[fact_items, fact_ordenes, dim_clientes, dim_productos],
    agent_type="tool-calling",    # ← cambiamos esto
    verbose=True,
    allow_dangerous_code=True
)

print("✅ Agente creado correctamente")

✅ Agente creado correctamente


### 🛠️ Configuración de un Agente Especializado (Restringido)

Para que el agente no responda cosas como "¿Dónde queda Areguá?", vamos a crear un nuevo agente con un **prefijo personalizado**.

In [ ]:
from langchain_experimental.agents import create_pandas_dataframe_agent
from langchain_mistralai import ChatMistralAI

# --- CONFIGURACIÓN PROFESIONAL DEL AGENTE ---

# 1. Definición del Motor (LLM)
llm = ChatMistralAI(model="mistral-small-latest", temperature=0)

# 2. Definición del Prefijo (System Prompt)
# En entornos profesionales, centralizamos las instrucciones de comportamiento en una variable de sistema.
prefix = """
Eres un Analista de Datos especializado en el dataset de ventas de la empresa.
Tu objetivo es EXAMINAR y extraer conclusiones únicamente de los DataFrames proporcionados.

REGLAS DE OPERACIÓN:
1. Si la consulta del usuario no tiene relación con los datos de las tablas (ventas, clientes, productos),
   responde exactamente: "Lo siento, mi base de conocimientos está limitada al dataset de ventas y no puedo responder esa pregunta".
2. Bajo ninguna circunstancia respondas preguntas de cultura general, geografía o temas personales.
3. Si los datos necesarios para responder no están presentes en las tablas, indícalo claramente.
"""

# 3. Instanciación del Agente
# Aplicamos el estándar profesional de Python: argumento=variable_local
agente_especialista = create_pandas_dataframe_agent(
    llm=llm,
    df=[fact_items, fact_ordenes, dim_clientes, dim_productos],
    agent_type="tool-calling",
    prefix=prefix,
    verbose=True,
    allow_dangerous_code=True
)

print("✅ Agente Especialista configurado bajo estándares profesionales.")

✅ Agente Especialista configurado bajo estándares profesionales.


### 🧪 Prueba de Restricción de Seguridad
Ejecuta esta celda para verificar si el agente respeta el `prefix` profesional.

In [ ]:
# Forzamos una pregunta fuera de contexto para probar el System Prompt
test_fuera_de_contexto = "¿Cuáles son los feriados de mayo?"

print(f"Pregunta de prueba: {test_fuera_de_contexto}\n")
resultado = agente_especialista.invoke(test_fuera_de_contexto)
print(f"Respuesta del Agente: {resultado['output']}")

Pregunta de prueba: ¿Cuáles son los feriados de mayo?



> Entering new AgentExecutor chain...
Lo siento, mi base de conocimientos está limitada al dataset de ventas y no puedo responder esa pregunta.

> Finished chain.
Respuesta del Agente: Lo siento, mi base de conocimientos está limitada al dataset de ventas y no puedo responder esa pregunta.


### 🔍 Análisis Técnico del Código (Estilo Senior)

1.  **`prefix = """..."""`**:
    *   **Para qué sirve**: Define el *System Message*. Es el ancla de personalidad del modelo. Profesionalmente, se usa para evitar que la IA alucine o responda cosas fuera de contexto.

2.  **`create_pandas_dataframe_agent(...)`**:
    *   **Para qué sirve**: Es un *Constructor*. Es la función que ensambla el LLM, las herramientas de Python y el acceso a tus archivos en un solo objeto inteligente.

3.  **`llm=llm`**:
    *   **Para qué sirve**: Inyección de Dependencias. Le estamos diciendo al agente: "Usa este motor de Mistral como tu cerebro". El primer `llm` es el nombre del parámetro que espera la función; el segundo es tu variable.

4.  **`df=[...]`**:
    *   **Para qué sirve**: Definición del Entorno. Al pasarle una lista de DataFrames, LangChain le genera al LLM un mapa de tus tablas para que sepa que `fact_ordenes` se puede unir con `dim_clientes`.

5.  **`agent_type="tool-calling"`**:
    *   **Para qué sirve**: Selección de Arquitectura. Es el método más robusto actualmente (sustituye a `zero-shot-react`). Permite que el modelo llame a la función de Python de forma mucho más precisa y rápida.

6.  **`prefix=prefix`**:
    *   **Para qué sirve**: Sobrescritura del Comportamiento Base. Por defecto, el agente es libre. Al pasarle tu variable `prefix`, estás sobreescribiendo sus reglas iniciales con tu contrato de "Solo ventas".

7.  **`verbose=True`**:
    *   **Para qué sirve**: Trazabilidad y Auditoría. En producción, esto nos permite ver los logs y entender exactamente qué código generó el agente y por qué falló o acertó.

8.  **`allow_dangerous_code=True`**:
    *   **Para qué sirve**: Declaración de Consentimiento. Como medida de seguridad, LangChain te obliga a confirmar que eres consciente de que la IA está ejecutando código generado dinámicamente en tu servidor.

Celda 8 — Primera pregunta al agente


In [ ]:
respuesta = agente.invoke("¿Cuantas ordenes hay en total?")
print(respuesta["output"])



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "# Contar el número total de órdenes únicas en df1 (tabla de órdenes)\ntotal_ordenes = df1['id_orden'].nunique()\ntotal_ordenes"}`


2Hay **2 órdenes** en total.

> Finished chain.
Hay **2 órdenes** en total.


Celda 9 — Preguntas de prueba


In [ ]:
preguntas = [
    "¿Cual es el producto mas vendido en cantidad?",
    "¿Que cliente genero mas ingresos en total?",
    "¿Cuantas ordenes fueron canceladas?",
]

for pregunta in preguntas:
    print(f"\n🔎 {pregunta}")
    print("-" * 50)
    respuesta = agente.invoke(pregunta)
    print(f"✅ {respuesta['output']}")
    print()


🔎 ¿Cual es el producto mas vendido en cantidad?
--------------------------------------------------


> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Unir los dataframes para obtener la información necesaria\nmerged_df = df1.merge(df2, on='id_orden')\nmerged_df = merged_df.merge(df3, on='id_cliente')\nmerged_df = merged_df.merge(df4, on='id_producto')\n\n# Agrupar por producto y sumar la cantidad vendida\nproduct_sales = merged_df.groupby('producto')['cantidad'].sum()\n\n# Encontrar el producto más vendido\nmost_sold_product = product_sales.idxmax()\nmost_sold_quantity = product_sales.max()\n\nmost_sold_product, most_sold_quantity"}`


('Mouse', 5)El producto más vendido en cantidad es **Mouse**, con un total de **5 unidades vendidas**.

> Finished chain.
✅ El producto más vendido en cantidad es **Mouse**, con un total de **5 unidades vendidas**.


🔎 ¿Que cliente genero mas ingresos en total?
---------------------------------

Celda 10 — Chat interactivo


In [ ]:
print("🤖 Agente ESPECIALISTA listo (Restringido al dataset). Escribí 'salir' para terminar.\n")

while True:
    pregunta = input("Vos: ")

    if pregunta.lower() == "salir":
        print("👋 Cerrando el agente.")
        break

    if pregunta.strip() == "":
        continue

    # CAMBIO CRÍTICO: Usamos 'agente_especialista' en lugar de 'agente'
    respuesta = agente_especialista.invoke(pregunta)
    print(f"\n🤖 Agente: {respuesta['output']}\n")

🤖 Agente ESPECIALISTA listo (Restringido al dataset). Escribí 'salir' para terminar.



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "import pandas as pd\n\n# Datos de ejemplo para los DataFrames (basados en la información proporcionada)\ndf_ventas = pd.DataFrame({\n    'id_orden': [501, 502],\n    'id_producto': [101, 102],\n    'cantidad': [1, 5],\n    'precio': [1000, 20]\n})\n\n# Contar el número de ventas (órdenes únicas)\nnumero_ventas = df_ventas['id_orden'].nunique()\nnumero_ventas"}`


2La empresa tuvo **2 ventas** registradas en el dataset proporcionado.

> Finished chain.

🤖 Agente: La empresa tuvo **2 ventas** registradas en el dataset proporcionado.



> Entering new AgentExecutor chain...
Lo siento, mi base de conocimientos está limitada al dataset de ventas y no puedo responder esa pregunta.

> Finished chain.

🤖 Agente: Lo siento, mi base de conocimientos está limitada al dataset de ventas y no puedo responder esa pregunta.



> Ent

Celda 11 — Instalar ChromaDB


In [ ]:
!pip install chromadb langchain-chroma

## 🧠 RAG — Retrieval-Augmented Generation

A partir de acá implementamos el RAG real.
El agente ya no ejecuta código sobre los DataFrames —
ahora busca información relevante en una base de datos vectorial
y se la pasa a Mistral como contexto para responder.

Pipeline completo:
1. Convertir los datos en texto legible (Documents)
2. Transformar ese texto en vectores numéricos (Embeddings)
3. Indexar los vectores en ChromaDB
4. Crear el Retriever (el buscador semántico)
5. Conectar el Retriever al agente

### 📄 Paso 1 — Importaciones RAG


### 🔄 Paso 2 — Convertir datos a Documents


### 🔢 Paso 3 — Embeddings e indexación en ChromaDB

### 🔍 Paso 4 — Crear el Retriever

### 🤖 Paso 5 — Conectar RAG al agente